# Wind Forecast — Benchmark v2 (Colab GPU)

Quy trình: clone repo → đồng bộ lên Drive → nạp kernel `mamba-ssm` (chưa có thì build) →
chạy nền → kết quả tự chép về `MyDrive/WIND_FORECAST_DATA/results/` mỗi 5 phút.

**Bật GPU:** Runtime → Change runtime type → T4 GPU. Chạy lần lượt từ trên xuống.

## 1. Drive + GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch, os, shutil, glob, subprocess, time
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
DRIVE = '/content/drive/MyDrive/WIND_FORECAST_DATA'
W = '/content/wind_forecast'

## 2. Lấy code + dữ liệu, lấy lại kết quả cũ từ Drive (để chạy tiếp, không chạy lại)

In [ ]:
shutil.rmtree(W, ignore_errors=True)
subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/phanduyhoang22/wind_forecast.git', W], check=True)
print(subprocess.run(['git', '-C', W, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
for sub, pat in (('scripts', '*.py'), ('data_v2', '*.npz')):
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)
    for f in glob.glob(f'{W}/{sub}/{pat}'):
        shutil.copy(f, f'{DRIVE}/{sub}/')
R = f'{DRIVE}/results'
os.makedirs(R, exist_ok=True)
for f in glob.glob(f'{R}/*.csv'):
    shutil.copy(f, f'{W}/results/')
for sub in ('preds', 'models'):
    if os.path.isdir(f'{R}/{sub}'):
        shutil.copytree(f'{R}/{sub}', f'{W}/results/{sub}', dirs_exist_ok=True)
os.chdir(W)
print(sorted(os.listdir('results')))

## 3. Kernel `mamba-ssm` (selective_scan, chỉ cho T4 / sm_75)

Có bản build khớp phiên bản torch trên Drive thì chép về (vài giây). Không có thì build
(~25 phút, C++20 vì header torch 2.14 yêu cầu) rồi lưu lên Drive cho lần sau.

In [ ]:
KDIR = '/content/ss_build'
os.makedirs(KDIR, exist_ok=True)
kfile = f'{DRIVE}/kernels/selective_scan_cuda_sm75_torch{torch.__version__.split("+")[0]}.so'
if os.path.exists(kfile):
    shutil.copy(kfile, f'{KDIR}/selective_scan_cuda.so')
    print('Nap kernel tu Drive:', kfile)
else:
    print('Chua co kernel cho torch nay -> build (~25 phut)')
    subprocess.run('pip install -q ninja', shell=True, check=True)
    subprocess.run('rm -rf /content/mamba && git clone -q --depth 1 https://github.com/state-spaces/mamba.git /content/mamba',
                   shell=True, check=True)
    S = '/content/mamba/csrc/selective_scan'
    srcs = ['selective_scan.cpp'] + sorted(f for f in os.listdir(S) if f.endswith('.cu'))
    flags = ['-O3', '-std=c++20', '-U__CUDA_NO_HALF_OPERATORS__', '-U__CUDA_NO_HALF_CONVERSIONS__',
             '-U__CUDA_NO_BFLOAT16_OPERATORS__', '-U__CUDA_NO_BFLOAT16_CONVERSIONS__',
             '-U__CUDA_NO_BFLOAT162_OPERATORS__', '-U__CUDA_NO_BFLOAT162_CONVERSIONS__',
             '--expt-relaxed-constexpr', '--expt-extended-lambda', '--use_fast_math',
             '-gencode', 'arch=compute_75,code=sm_75']
    os.environ['MAX_JOBS'] = '2'
    from torch.utils.cpp_extension import load
    load(name='selective_scan_cuda', sources=[f'{S}/{f}' for f in srcs], build_directory=KDIR,
         extra_cflags=['-O3', '-std=c++20'], extra_cuda_cflags=flags, verbose=False)
    os.makedirs(f'{DRIVE}/kernels', exist_ok=True)
    shutil.copy(f'{KDIR}/selective_scan_cuda.so', kfile)
    print('Build xong, da luu', kfile)

## 4. Kiểm định: self-test mô hình (kernel CUDA khớp bản tham chiếu) + baseline

In [ ]:
!cd /content/wind_forecast && PYTHONPATH=/content/ss_build python scripts/06_models.py
!cd /content/wind_forecast && python scripts/04_data_pipeline.py 2>&1 | tail -7

## 5. Chạy nền

Đổi `CMD` cho từng giai đoạn. Kết quả ghi nối tiếp; chạy lại ô này sau khi Colab ngắt sẽ
tự bỏ qua lượt đã xong.

In [ ]:
NN = 'LSTM,Transformer,Mamba,PatchTST,DLinear'
CMD = f'scripts/07_run_benchmark.py --models {NN} --seeds 5 --epochs 50 --patience 10 --out results/runs.csv'
# CMD = 'scripts/09_global_loso.py --models Mamba,LSTM --seeds 3 --out results/loso.csv'
# CMD = 'scripts/10_shap.py --sites A,B,C,D --n-explain 1000 --n-background 200'
LOG = 'results/job.log'
RUN = f'cd {W} && PYTHONPATH=/content/ss_build python -u {CMD} >> {LOG} 2>&1; touch results/job.done'
SYNC = (f'while true; do D={DRIVE}/results; cp {W}/results/*.csv {W}/results/*.log $D/ 2>/dev/null; '
        f'for s in preds preds_loso models; do mkdir -p $D/$s; cp -u {W}/results/$s/* $D/$s/ 2>/dev/null; done; '
        'sleep 300; done')
if os.path.exists(f'{W}/results/job.done'): os.remove(f'{W}/results/job.done')
job = subprocess.Popen(['bash', '-c', RUN]); sync = subprocess.Popen(['bash', '-c', SYNC])
print('job', job.pid, '| sync', sync.pid)

## 6. Tiến độ (chạy lại bất cứ lúc nào)

In [ ]:
print('XONG' if os.path.exists(f'{W}/results/job.done') else 'dang chay')
!tail -n 12 /content/wind_forecast/results/job.log
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

## 7. Lưu về Drive ngay (trước khi đóng phiên)

In [ ]:
subprocess.run(f'D={DRIVE}/results; cp {W}/results/*.csv {W}/results/*.log $D/; '
               f'for s in preds preds_loso models; do mkdir -p $D/$s; cp -u {W}/results/$s/* $D/$s/ 2>/dev/null; done',
               shell=True)
print(sorted(os.listdir(f'{DRIVE}/results')))